[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/074.%20The%20VRP%20with%20Backhauls%20%28VRPB%29/P74-Tier-3_executed.ipynb)

---


# 74. The Vehicle Routing Problem with Backhauls

## Tier 3: A Dragonfly-Style Swarm Search

### Purpose and scope

This notebook applies a population-based Dragonfly Algorithm to a small Vehicle Routing Problem with Backhauls (VRPB) instance. The algorithm searches for a low-distance customer sequence while preserving the required delivery-before-pickup order. It represents each dragonfly as a feasible permutation and adapts the five Dragonfly behaviors—separation, alignment, cohesion, food attraction, and enemy distraction—to this discrete search space.

The instance has one vehicle and one route that starts and ends at a depot. Each delivery customer must be visited before every pickup customer. Vehicle capacity is checked throughout the route. The example is deliberately small; this implementation is an educational metaheuristic, not a guarantee of a globally optimal solution.

**Inputs:** customer counts, demands, vehicle capacity, a depot-inclusive distance matrix, and Dragonfly parameters.  
**Outputs:** the best feasible sequence found, its total travel distance, an iteration history, and diagnostic visualizations.

### Modeling assumptions

- Customer indices are consecutive integers: the depot is node 0, deliveries are nodes 1 through `num_deliveries`, and pickups follow them.
- The vehicle leaves the depot carrying the total delivery demand. Deliveries reduce its load; pickups increase it. The load must remain between zero and vehicle capacity after each visit.
- All deliveries precede all pickups. The customer order within each group may vary.
- Travel cost is the sum of the distances along the route from the depot, through each customer, and back to the depot. Distances are supplied in a matrix; no coordinates or travel-time model are assumed.
- The example represents a single vehicle and a single route. Fleet-size decisions, time windows, service times, split deliveries, and multiple depots are outside its scope.
- The search is stochastic. No random seed is set, so separate executions may produce different search histories and may return different equally good routes.

### VRPB model

Let \(D\) be the set of delivery customers, \(P\) the set of pickup customers, and \(V = \{0\} \cup D \cup P\) the nodes, with depot 0. A route is a permutation \(\pi = (\pi_1,\ldots,\pi_n)\) of all customers, where \(n=|D|+|P|\). The objective is to minimize total distance:

$$
\min_{\pi}\quad c_{0,\pi_1} + \sum_{k=1}^{n-1} c_{\pi_k,\pi_{k+1}} + c_{\pi_n,0}.
$$

The sequence must visit every customer exactly once and satisfy the precedence condition

$$
\max_{d\in D} \operatorname{pos}_{\pi}(d)
<
\min_{p\in P} \operatorname{pos}_{\pi}(p),
$$

when both customer groups are nonempty. Here, \(c_{ij}\) is the distance from node \(i\) to node \(j\), and \(\operatorname{pos}_{\pi}(i)\) is the position of customer \(i\) in the route.

For capacity, let \(q_i\) be the demand of customer \(i\), interpreted as a delivery quantity for \(i\in D\) and a pickup quantity for \(i\in P\). The load after serving the first \(k\) customers is

$$
L_0 = \sum_{d\in D} q_d,
\qquad
L_k = L_{k-1} - q_{\pi_k}\; (\pi_k\in D),
\qquad
L_k = L_{k-1} + q_{\pi_k}\; (\pi_k\in P),
$$

with \(0\le L_k\le Q\) for every \(k\), where \(Q\) is vehicle capacity. The code checks these constraints in `calculate_fitness`; an infeasible sequence receives infinite fitness.

### Search representation and Dragonfly behaviors

The continuous Dragonfly Algorithm uses position and velocity vectors. A customer sequence, however, is a permutation: customer labels are categories, so subtracting their numerical labels would give arbitrary, meaningless movement. Here, each dragonfly's position remains a customer sequence, while behavior vectors use **rank coordinates**: the coordinate for a customer is that customer's place in the sequence. This gives differences a consistent interpretation across permutations.

For a dragonfly with rank vector \(x\), velocity \(v\), and neighbors \(N\), the behavior vectors are

$$
\begin{aligned}
S &= \sum_{y\in N}(x-y), &
A &= \frac{1}{|N|}\sum_{y\in N}v_y,\\
C &= \frac{1}{|N|}\sum_{y\in N}y-x, &
F &= x_{\mathrm{food}}-x,\\
E &= x-x_{\mathrm{enemy}}.
\end{aligned}
$$

Separation discourages crowding, alignment uses neighboring velocities, and cohesion points toward the neighbors' average rank vector. Food attraction points toward the best known route; enemy distraction points away from a poor route. When there are no neighbors, the first three vectors are zero. The weighted step and velocity update are

$$
\Delta v = w_s S + w_a A + w_c C + w_f F + w_e E + w_I v,
\qquad
v \leftarrow \Delta v.
$$

The continuous step is converted into a discrete move: each customer coordinate has a probability, capped at 1, of triggering a swap in the permutation. A precedence repair then places deliveries before pickups while retaining the order within each group. The repair preserves the required constraint, though it can change the route produced by the swaps. This is a discrete adaptation of the Dragonfly search pattern, not a claim that the continuous equations transfer exactly to permutations.

### Algorithm flow

1. Create a population of random feasible routes and initialize their velocities.
2. Identify the current food (best) and enemy (worst) routes.
3. For each dragonfly, find nearby routes using Hamming distance between customer sequences and calculate the five behavioral vectors in rank coordinates.
4. Update its velocity, probabilistically swap customers, and repair delivery-before-pickup precedence.
5. Evaluate the new route, update personal and global bests, and record the best and mean feasible fitness.
6. Reduce the swap step size at the specified intervals and report the search history.

A **state** here is one route together with its fitness and velocity. The best-so-far route is retained separately, so a later move cannot erase the best solution already found. Unlike dynamic programming, this method does not use a Bellman recurrence: it repeatedly updates a population of candidate states. With population size \(m\), \(T\) iterations, and \(n\) customers, fitness evaluation is \(O(n)\) per candidate, while neighborhood comparisons and rank-based behavior calculations are approximately \(O(m^2 n)\) per iteration. The overall search cost is therefore approximately \(O(Tm^2n)\), in addition to initialization.

### Interpreting the results

The best fitness is the total route distance for the best feasible sequence found, not a proof of global optimality. The best-fitness history should be non-increasing because the global best is retained. The mean population fitness may rise or fall as dragonflies explore. The example's parameter-sensitivity experiment changes selected search settings; it does not establish that one configuration is generally superior. Since the process is stochastic, comparisons across configurations would be more reliable over multiple controlled runs, which this notebook does not perform.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Keep plots visually consistent while leaving the algorithm's random search unseeded.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully for the Dragonfly-style search.")

In [ ]:
@dataclass
class VRPBInstance:
    """Data and index sets for a single-route VRPB instance."""

    num_deliveries: int
    num_pickups: int
    delivery_demands: List[float]
    pickup_demands: List[float]
    capacity: float
    distance_matrix: np.ndarray

    def __post_init__(self):
        if self.num_deliveries < 0 or self.num_pickups < 0:
            raise ValueError("Customer counts must be nonnegative.")
        if len(self.delivery_demands) != self.num_deliveries:
            raise ValueError("There must be one demand for each delivery customer.")
        if len(self.pickup_demands) != self.num_pickups:
            raise ValueError("There must be one demand for each pickup customer.")
        if self.capacity < 0:
            raise ValueError("Vehicle capacity must be nonnegative.")

        self.num_customers = self.num_deliveries + self.num_pickups
        expected_shape = (self.num_customers + 1, self.num_customers + 1)
        self.distance_matrix = np.asarray(self.distance_matrix, dtype=float)
        if self.distance_matrix.shape != expected_shape:
            raise ValueError(f"Distance matrix must have shape {expected_shape}.")
        if np.any(self.distance_matrix < 0):
            raise ValueError("Distances must be nonnegative.")
        if any(demand < 0 for demand in self.delivery_demands + self.pickup_demands):
            raise ValueError("Customer demands must be nonnegative.")

        self.delivery_indices = list(range(1, self.num_deliveries + 1))
        self.pickup_indices = list(
            range(self.num_deliveries + 1, self.num_customers + 1)
        )
        self.all_customer_indices = self.delivery_indices + self.pickup_indices


@dataclass
class Dragonfly:
    """One candidate route, its rank-coordinate velocity, and personal best."""

    position: List[int]
    velocity: List[float]
    fitness: float = float("inf")
    best_position: List[int] = field(default_factory=list)
    best_fitness: float = float("inf")

    def update_personal_best(self):
        """Retain the best route this dragonfly has visited."""
        if self.fitness < self.best_fitness:
            self.best_fitness = self.fitness
            self.best_position = self.position.copy()


@dataclass
class DragonflyParameters:
    """Population, iteration, behavior-weight, and discrete-step settings."""

    population_size: int = 20
    max_iterations: int = 100
    separation_weight: float = 0.1
    alignment_weight: float = 0.1
    cohesion_weight: float = 0.1
    food_weight: float = 0.2
    enemy_weight: float = 0.1
    inertia_weight: float = 0.9
    step_size: float = 0.1
    adaptive_factor: float = 0.95

print("VRPB data structures and Dragonfly parameters are ready.")

In [ ]:
def create_concrete_example() -> VRPBInstance:
    """Create the specified example with four deliveries and three pickups."""
    num_deliveries = 4
    num_pickups = 3
    delivery_demands = [5, 4, 6, 3]
    pickup_demands = [4, 5, 3]
    capacity = 20

    # Rows and columns are depot 0, deliveries 1–4, and pickups 5–7.
    distance_matrix = np.array(
        [
            [0, 3, 4, 5, 6, 8, 7, 9],
            [3, 0, 2, 3, 4, 6, 5, 7],
            [4, 2, 0, 2, 3, 5, 4, 6],
            [5, 3, 2, 0, 2, 4, 3, 5],
            [6, 4, 3, 2, 0, 3, 2, 4],
            [8, 6, 5, 4, 3, 0, 2, 3],
            [7, 5, 4, 3, 2, 2, 0, 2],
            [9, 7, 6, 5, 4, 3, 2, 0],
        ],
        dtype=float,
    )

    return VRPBInstance(
        num_deliveries=num_deliveries,
        num_pickups=num_pickups,
        delivery_demands=delivery_demands,
        pickup_demands=pickup_demands,
        capacity=capacity,
        distance_matrix=distance_matrix,
    )


instance = create_concrete_example()
print(
    f"Created a VRPB instance with {instance.num_deliveries} deliveries "
    f"and {instance.num_pickups} pickups."
)
print(f"Vehicle capacity: {instance.capacity}")
print(f"Total delivery demand: {sum(instance.delivery_demands)}")
print(f"Total pickup demand: {sum(instance.pickup_demands)}")

In [ ]:
class DragonflyVRPBOptimizer:
    """Dragonfly-style population search for a single-route VRPB instance."""

    def __init__(
        self,
        instance: VRPBInstance,
        params: Optional[DragonflyParameters] = None,
    ):
        self.instance = instance
        self.params = params or DragonflyParameters()
        if self.params.population_size < 1:
            raise ValueError("Population size must be at least one.")
        if self.params.max_iterations < 0:
            raise ValueError("Maximum iterations must be nonnegative.")
        self.dragonflies: List[Dragonfly] = []
        self.food_source: List[int] = []
        self.enemy_source: List[int] = []
        self.best_fitness_history: List[float] = []
        self.avg_fitness_history: List[float] = []

    def calculate_fitness(self, position: List[int]) -> float:
        """Return route distance, or infinity when a route is infeasible."""
        expected = self.instance.all_customer_indices
        if len(position) != len(expected) or set(position) != set(expected):
            return float("inf")

        delivery_positions = [position.index(d) for d in self.instance.delivery_indices]
        pickup_positions = [position.index(p) for p in self.instance.pickup_indices]
        if delivery_positions and pickup_positions:
            if max(delivery_positions) >= min(pickup_positions):
                return float("inf")

        # The vehicle starts with all delivery goods; each visit changes its load.
        load = float(sum(self.instance.delivery_demands))
        if load > self.instance.capacity:
            return float("inf")
        for customer in position:
            if customer in self.instance.delivery_indices:
                demand = self.instance.delivery_demands[customer - 1]
                load -= demand
            else:
                pickup_index = customer - self.instance.num_deliveries - 1
                demand = self.instance.pickup_demands[pickup_index]
                load += demand
            if load < 0 or load > self.instance.capacity:
                return float("inf")

        distances = self.instance.distance_matrix
        total_distance = distances[0, position[0]]
        for first, second in zip(position, position[1:]):
            total_distance += distances[first, second]
        total_distance += distances[position[-1], 0]
        return float(total_distance)

    def generate_feasible_position(self) -> List[int]:
        """Shuffle within each customer group, then impose backhaul precedence."""
        deliveries = self.instance.delivery_indices.copy()
        pickups = self.instance.pickup_indices.copy()
        random.shuffle(deliveries)
        random.shuffle(pickups)
        return deliveries + pickups

    def initialize_swarm(self):
        """Create feasible candidate routes and initialize their velocities."""
        self.dragonflies = []
        for _ in range(self.params.population_size):
            position = self.generate_feasible_position()
            dragonfly = Dragonfly(
                position=position,
                velocity=[0.0] * len(position),
            )
            dragonfly.fitness = self.calculate_fitness(position)
            dragonfly.update_personal_best()
            self.dragonflies.append(dragonfly)
        self.update_food_enemy_sources()

    def update_food_enemy_sources(self):
        """Set food to the best personal route and enemy to the worst current route."""
        if not self.dragonflies:
            self.food_source = []
            self.enemy_source = []
            return

        best_dragonfly = min(self.dragonflies, key=lambda dragonfly: dragonfly.best_fitness)
        self.food_source = best_dragonfly.best_position.copy()
        worst_dragonfly = max(self.dragonflies, key=lambda dragonfly: dragonfly.fitness)
        self.enemy_source = worst_dragonfly.position.copy()

    def _rank_vector(self, position: List[int]) -> List[float]:
        """Index each customer's route rank by its consecutive customer index."""
        ranks = [0.0] * len(position)
        for rank, customer in enumerate(position):
            ranks[customer - 1] = float(rank)
        return ranks

    def calculate_separation(
        self, dragonfly: Dragonfly, neighbors: List[Dragonfly]
    ) -> List[float]:
        """Point away from neighboring rank vectors to reduce crowding."""
        current = self._rank_vector(dragonfly.position)
        separation = [0.0] * len(current)
        for neighbor in neighbors:
            neighbor_ranks = self._rank_vector(neighbor.position)
            for customer_index in range(len(current)):
                separation[customer_index] += (
                    current[customer_index] - neighbor_ranks[customer_index]
                )
        return separation

    def calculate_alignment(
        self, dragonfly: Dragonfly, neighbors: List[Dragonfly]
    ) -> List[float]:
        """Use the mean neighboring velocity in customer-rank coordinates."""
        if not neighbors:
            return [0.0] * len(dragonfly.velocity)
        return [
            sum(neighbor.velocity[i] for neighbor in neighbors) / len(neighbors)
            for i in range(len(dragonfly.velocity))
        ]

    def calculate_cohesion(
        self, dragonfly: Dragonfly, neighbors: List[Dragonfly]
    ) -> List[float]:
        """Point toward the neighbors' mean rank vector."""
        if not neighbors:
            return [0.0] * len(dragonfly.position)
        current = self._rank_vector(dragonfly.position)
        neighbor_ranks = [self._rank_vector(neighbor.position) for neighbor in neighbors]
        return [
            sum(ranks[i] for ranks in neighbor_ranks) / len(neighbors) - current[i]
            for i in range(len(current))
        ]

    def calculate_food_attraction(self, dragonfly: Dragonfly) -> List[float]:
        """Point toward the rank vector of the best known route."""
        if not self.food_source:
            return [0.0] * len(dragonfly.position)
        current = self._rank_vector(dragonfly.position)
        food = self._rank_vector(self.food_source)
        return [food[i] - current[i] for i in range(len(current))]

    def calculate_enemy_distraction(self, dragonfly: Dragonfly) -> List[float]:
        """Point away from the rank vector of the worst current route."""
        if not self.enemy_source:
            return [0.0] * len(dragonfly.position)
        current = self._rank_vector(dragonfly.position)
        enemy = self._rank_vector(self.enemy_source)
        return [current[i] - enemy[i] for i in range(len(current))]

    def update_position(self, dragonfly: Dragonfly) -> List[int]:
        """Combine swarm behaviors, then create a precedence-feasible permutation."""
        neighbors = []
        distance_threshold = 3
        for other in self.dragonflies:
            if other is dragonfly:
                continue
            hamming_distance = sum(
                first != second
                for first, second in zip(dragonfly.position, other.position)
            )
            if hamming_distance <= distance_threshold:
                neighbors.append(other)

        separation = self.calculate_separation(dragonfly, neighbors)
        alignment = self.calculate_alignment(dragonfly, neighbors)
        cohesion = self.calculate_cohesion(dragonfly, neighbors)
        food_attraction = self.calculate_food_attraction(dragonfly)
        enemy_distraction = self.calculate_enemy_distraction(dragonfly)

        step_vector = [0.0] * len(dragonfly.position)
        for i in range(len(step_vector)):
            step_vector[i] = (
                self.params.separation_weight * separation[i]
                + self.params.alignment_weight * alignment[i]
                + self.params.cohesion_weight * cohesion[i]
                + self.params.food_weight * food_attraction[i]
                + self.params.enemy_weight * enemy_distraction[i]
                + self.params.inertia_weight * dragonfly.velocity[i]
            )

        dragonfly.velocity = step_vector.copy()
        return self.discrete_position_update(dragonfly.position, step_vector)

    def discrete_position_update(
        self, current_position: List[int], step_vector: List[float]
    ) -> List[int]:
        """Use step magnitudes to trigger permutation swaps, then repair precedence."""
        new_position = current_position.copy()
        for customer_index, step in enumerate(step_vector):
            probability = min(1.0, abs(step) * self.params.step_size)
            if random.random() < probability and len(new_position) > 1:
                current_rank = new_position.index(customer_index + 1)
                other_rank = random.randrange(len(new_position) - 1)
                if other_rank >= current_rank:
                    other_rank += 1
                new_position[current_rank], new_position[other_rank] = (
                    new_position[other_rank],
                    new_position[current_rank],
                )
        return self.repair_precedence_constraint(new_position)

    def repair_precedence_constraint(self, position: List[int]) -> List[int]:
        """Group deliveries before pickups without changing within-group order."""
        delivery_set = set(self.instance.delivery_indices)
        pickup_set = set(self.instance.pickup_indices)
        deliveries = [customer for customer in position if customer in delivery_set]
        pickups = [customer for customer in position if customer in pickup_set]
        return deliveries + pickups

    def optimize(self) -> Tuple[List[int], float, List[float]]:
        """Run the swarm search and return the best route, cost, and history."""
        print("Starting the Dragonfly-style VRPB search...")
        self.best_fitness_history = []
        self.avg_fitness_history = []
        self.initialize_swarm()

        best_dragonfly = min(self.dragonflies, key=lambda dragonfly: dragonfly.fitness)
        best_global_position = best_dragonfly.position.copy()
        best_global_fitness = best_dragonfly.fitness

        for iteration in range(self.params.max_iterations):
            for dragonfly in self.dragonflies:
                new_position = self.update_position(dragonfly)
                new_fitness = self.calculate_fitness(new_position)
                dragonfly.position = new_position
                dragonfly.fitness = new_fitness
                dragonfly.update_personal_best()

                if new_fitness < best_global_fitness:
                    best_global_fitness = new_fitness
                    best_global_position = new_position.copy()

            self.update_food_enemy_sources()
            feasible_fitness = [
                dragonfly.fitness
                for dragonfly in self.dragonflies
                if np.isfinite(dragonfly.fitness)
            ]
            if not feasible_fitness:
                raise RuntimeError("The swarm contains no feasible route.")

            self.best_fitness_history.append(best_global_fitness)
            self.avg_fitness_history.append(float(np.mean(feasible_fitness)))

            if iteration % 25 == 0 or iteration == self.params.max_iterations - 1:
                print(
                    f"  Iteration {iteration:3d}: "
                    f"best distance = {best_global_fitness:.2f}, "
                    f"mean distance = {self.avg_fitness_history[-1]:.2f}"
                )

            if iteration > 0 and iteration % 50 == 0:
                self.params.step_size *= self.params.adaptive_factor

        return best_global_position, best_global_fitness, self.best_fitness_history

print("Dragonfly-style optimizer is ready.")

In [ ]:
optimizer = DragonflyVRPBOptimizer(instance)
start_time = time.perf_counter()
best_position, best_fitness, fitness_history = optimizer.optimize()
computation_time = (time.perf_counter() - start_time) * 1000

print("\nSearch summary")
print("=" * 50)
print(f"Best feasible route: {best_position}")
print(f"Total route distance: {best_fitness:.2f}")
print(f"Elapsed time: {computation_time:.2f} ms")
print(f"Iterations: {optimizer.params.max_iterations}")
print(f"Population size: {optimizer.params.population_size}")

# Check that the reported best route satisfies the same rules used for fitness.
assert np.isfinite(optimizer.calculate_fitness(best_position))
assert optimizer.calculate_fitness(best_position) == best_fitness
assert set(best_position) == set(instance.all_customer_indices)
if instance.delivery_indices and instance.pickup_indices:
    assert max(best_position.index(d) for d in instance.delivery_indices) < min(
        best_position.index(p) for p in instance.pickup_indices
    )
print("Route coverage, precedence, capacity, and reported distance are verified.")

In [ ]:
def visualize_dragonfly_optimization(
    best_position: List[int],
    fitness_history: List[float],
    instance: VRPBInstance,
    optimizer: DragonflyVRPBOptimizer,
):
    """Plot convergence, route order, behavior weights, and route-leg distances."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle("Dragonfly-Style VRPB Search", fontsize=16, fontweight="bold")

    iterations = range(1, len(fitness_history) + 1)
    ax1.plot(iterations, fitness_history, label="Best-so-far distance", linewidth=2)
    ax1.plot(
        iterations,
        optimizer.avg_fitness_history,
        label="Mean feasible population distance",
        linestyle="--",
        alpha=0.8,
    )
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Route distance")
    ax1.set_title("Search history")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    if best_position:
        delivery_set = set(instance.delivery_indices)
        colors = [
            "tab:red" if customer in delivery_set else "tab:blue"
            for customer in best_position
        ]
        route_positions = list(range(1, len(best_position) + 1))
        ax2.bar(route_positions, [1] * len(best_position), color=colors, alpha=0.75)
        ax2.set_xlabel("Visit order")
        ax2.set_ylabel("Customer category")
        ax2.set_yticks([])
        ax2.set_xticks(route_positions)
        ax2.set_xticklabels([str(customer) for customer in best_position])
        ax2.set_title("Best route sequence (labels show customer indices)")
        from matplotlib.patches import Patch

        ax2.legend(
            handles=[
                Patch(facecolor="tab:red", label="Delivery"),
                Patch(facecolor="tab:blue", label="Pickup"),
            ]
        )
        ax2.grid(True, axis="x", alpha=0.3)

    factor_names = ["Separation", "Alignment", "Cohesion", "Food", "Enemy"]
    factor_weights = [
        optimizer.params.separation_weight,
        optimizer.params.alignment_weight,
        optimizer.params.cohesion_weight,
        optimizer.params.food_weight,
        optimizer.params.enemy_weight,
    ]
    ax3.bar(factor_names, factor_weights, alpha=0.75)
    ax3.set_xlabel("Behavioral factor")
    ax3.set_ylabel("Weight")
    ax3.set_title("Dragonfly behavior weights")
    ax3.grid(True, axis="y", alpha=0.3)

    route_nodes = [0] + best_position + [0]
    leg_distances = [
        instance.distance_matrix[first, second]
        for first, second in zip(route_nodes, route_nodes[1:])
    ]
    leg_labels = [f"{first} → {second}" for first, second in zip(route_nodes, route_nodes[1:])]
    ax4.bar(range(1, len(leg_distances) + 1), leg_distances, alpha=0.75)
    ax4.set_xlabel("Route leg")
    ax4.set_ylabel("Distance")
    ax4.set_xticks(range(1, len(leg_distances) + 1))
    ax4.set_xticklabels(leg_labels, rotation=45, ha="right")
    ax4.set_title("Distance by route leg")
    ax4.grid(True, axis="y", alpha=0.3)

    fig.tight_layout()
    plt.show()


visualize_dragonfly_optimization(best_position, fitness_history, instance, optimizer)

In [ ]:
def analyze_dragonfly_performance(
    best_position: List[int],
    best_fitness: float,
    fitness_history: List[float],
    optimizer: DragonflyVRPBOptimizer,
    computation_time_ms: float,
):
    """Report convergence, population diversity, evaluations, and settings."""
    print("Dragonfly-style search diagnostics")
    print("=" * 50)

    print("\nConvergence")
    if fitness_history:
        initial_fitness = fitness_history[0]
        final_fitness = fitness_history[-1]
        total_improvement = (
            (initial_fitness - final_fitness) / initial_fitness * 100
            if initial_fitness > 0
            else 0.0
        )
        halfway_index = (len(fitness_history) - 1) // 2
        halfway_fitness = fitness_history[halfway_index]
        early_improvement = (
            (initial_fitness - halfway_fitness) / initial_fitness * 100
            if initial_fitness > 0
            else 0.0
        )
        late_improvement = (
            (halfway_fitness - final_fitness) / halfway_fitness * 100
            if halfway_fitness > 0
            else 0.0
        )
        print(f"Initial recorded best distance: {initial_fitness:.2f}")
        print(f"Final best distance: {final_fitness:.2f}")
        print(f"Improvement across recorded iterations: {total_improvement:.2f}%")
        print(f"Improvement in the first part: {early_improvement:.2f}%")
        print(f"Improvement in the second part: {late_improvement:.2f}%")
        print("The best-so-far history is non-increasing by construction.")
    else:
        print("No iterations were requested; the initialized population supplies the best route.")

    print("\nFinal swarm diversity")
    feasible_positions = [
        dragonfly.position
        for dragonfly in optimizer.dragonflies
        if np.isfinite(dragonfly.fitness)
    ]
    pairwise_distances = []
    for first_index in range(len(feasible_positions)):
        for second_index in range(first_index + 1, len(feasible_positions)):
            pairwise_distances.append(
                sum(
                    first != second
                    for first, second in zip(
                        feasible_positions[first_index],
                        feasible_positions[second_index],
                    )
                )
            )
    diversity = float(np.mean(pairwise_distances)) if pairwise_distances else 0.0
    print(f"Mean pairwise Hamming distance: {diversity:.2f}")
    print("This is a descriptive measure of route variation, not a quality score.")

    print("\nComputational effort")
    evaluations = optimizer.params.population_size * (optimizer.params.max_iterations + 1)
    print(f"Fitness evaluations, including initialization: {evaluations}")
    print(f"Elapsed time: {computation_time_ms:.2f} ms")
    if computation_time_ms > 0:
        print(f"Approximate evaluations per second: {evaluations / (computation_time_ms / 1000):.0f}")
    print("Elapsed time depends on the machine and runtime environment.")

    print("\nSearch settings")
    print(f"Population size: {optimizer.params.population_size}")
    print(f"Maximum iterations: {optimizer.params.max_iterations}")
    print(f"Separation weight: {optimizer.params.separation_weight}")
    print(f"Alignment weight: {optimizer.params.alignment_weight}")
    print(f"Cohesion weight: {optimizer.params.cohesion_weight}")
    print(f"Food weight: {optimizer.params.food_weight}")
    print(f"Enemy weight: {optimizer.params.enemy_weight}")
    print(f"Inertia weight: {optimizer.params.inertia_weight}")
    print(f"Final step size: {optimizer.params.step_size}")
    print(f"Best route: {best_position}")
    print(f"Best route distance: {best_fitness:.2f}")


analyze_dragonfly_performance(
    best_position,
    best_fitness,
    fitness_history,
    optimizer,
    computation_time,
)

In [ ]:
def test_parameter_sensitivity(instance: VRPBInstance):
    """Compare selected settings on the same instance using single stochastic runs."""
    print("Parameter sensitivity experiment")
    print("=" * 50)
    print(
        "Each configuration is run once. Results are illustrative because the "
        "search is stochastic and no random seed is fixed."
    )

    test_configs = [
        {"name": "Default", "population_size": 20, "max_iterations": 100, "food_weight": 0.2},
        {"name": "Large population", "population_size": 30, "max_iterations": 100, "food_weight": 0.2},
        {"name": "More iterations", "population_size": 20, "max_iterations": 150, "food_weight": 0.2},
        {"name": "Higher food weight", "population_size": 20, "max_iterations": 100, "food_weight": 0.4},
        {"name": "Balanced settings", "population_size": 25, "max_iterations": 120, "food_weight": 0.3},
    ]
    results = []

    for config in test_configs:
        print(f"\nTesting {config['name']}")
        params = DragonflyParameters(
            population_size=config["population_size"],
            max_iterations=config["max_iterations"],
            food_weight=config["food_weight"],
        )
        test_optimizer = DragonflyVRPBOptimizer(instance, params)
        start_time = time.perf_counter()
        _, test_fitness, test_history = test_optimizer.optimize()
        elapsed_ms = (time.perf_counter() - start_time) * 1000
        improvement = (
            (test_history[0] - test_history[-1]) / test_history[0] * 100
            if test_history and test_history[0] > 0
            else 0.0
        )
        results.append(
            {
                "config": config["name"],
                "fitness": test_fitness,
                "improvement": improvement,
                "time_ms": elapsed_ms,
            }
        )
        print(f"Best route distance: {test_fitness:.2f}")
        print(f"Recorded best-distance improvement: {improvement:.2f}%")
        print(f"Elapsed time: {elapsed_ms:.2f} ms")

    print("\nConfiguration summary")
    print(f"{'Configuration':<22} {'Distance':>10} {'Improvement':>13} {'Time (ms)':>12}")
    print("-" * 60)
    for result in results:
        print(
            f"{result['config']:<22} {result['fitness']:>10.2f} "
            f"{result['improvement']:>12.2f}% {result['time_ms']:>12.2f}"
        )

    best_result = min(results, key=lambda result: result["fitness"])
    tied_results = [
        result["config"]
        for result in results
        if np.isclose(result["fitness"], best_result["fitness"], rtol=1e-9, atol=1e-9)
    ]
    print(f"\nLowest observed distance: {best_result['fitness']:.2f}")
    print(f"Configuration(s) attaining it in this experiment: {', '.join(tied_results)}")
    print("A single run does not establish a generally best parameter setting.")


test_parameter_sensitivity(instance)

## Summary and conclusions

The notebook formulates a single-route VRPB as a distance-minimization problem with customer-coverage, delivery-before-pickup, and vehicle-capacity requirements. The Dragonfly-style search keeps a population of routes, combines separation, alignment, cohesion, food attraction, and enemy distraction in rank coordinates, then converts those movements into permutation swaps and repairs precedence.

The reported best route is the best feasible candidate encountered during this stochastic run. It is not certified as globally optimal. A non-increasing best-so-far history is expected because the algorithm retains the best solution, while the population mean and diversity describe exploration rather than guaranteed progress. The parameter experiment changes population size, iteration count, or food weight one configuration at a time, but its single-run outcomes should be interpreted cautiously.

### Strengths and limitations

**Strengths**
- Population-based exploration maintains several candidate routes at once.
- The five behavioral components and adaptive swap step make the search logic visible.
- Explicit fitness checks make route coverage, precedence, and capacity part of the evaluated model.
- The same framework can represent additional constraints if the instance and feasibility checks are extended.

**Limitations**
- The discrete swaps are a practical adaptation of continuous Dragonfly movement; they do not inherit a global-optimality guarantee from the continuous formulation.
- Stochastic results can vary between runs, and the sensitivity experiment uses only one run per configuration.
- The example contains one vehicle and one route and omits operational details such as time windows and service times.
- Computational time is environment-dependent and should not be treated as a hardware-independent performance claim.

### Further study

Natural extensions include testing multiple random runs, increasing the instance size, or adding explicitly modeled operational constraints. Each extension should preserve the route representation and make its feasibility conditions part of the fitness evaluation. These are possible directions for further study, not procedures carried out in this notebook.